Here we set up a new Terra data table (`watershed_snv_run`) where we store the inputs and outputs of the Watershed-SV workflow.

This notebook should take about 1 minute to run.

## Setup

In [1]:
BiocManager::install('AnVILGCP')

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.21 (BiocManager 1.30.25), R 4.5.0 (2025-04-11)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'AnVILGCP'”
Old packages: 'alabaster.base', 'AnnotationHub', 'AnVIL', 'base64enc', 'BH',
  'bigrquery', 'BiocFileCache', 'BiocGenerics', 'BiocManager', 'BiocParallel',
  'blob', 'boot', 'broom', 'bslib', 'cli', 'clock', 'cluster', 'colorspace',
  'commonmark', 'cowplot', 'cpp11', 'credentials', 'crosstalk', 'curl',
  'data.table', 'DBI', 'dbplyr', 'DelayedArray', 'DESeq2', 'devtools',
  'diffobj', 'digest', 'downlit', 'dplyr', 'DT', 'dtplyr', 'edgeR', 'evaluate',
  'ExperimentHub', 'fansi', 'fitdistrplus', 'forcats', 'foreign', 'fs',
  'futile.logger', 'future', 'future.apply', 'gargle', 'generics'

In [2]:
library(AnVILGCP)
library(data.table)
library(jsonlite)

Similar to notebook 1, the `avtable()` function is used to import Terra data tables into R as a data frame.

In [3]:
anvil_file_tbl <- setDT(avtable('anvil_file'   ))
anno_meta_tbl  <- setDT(avtable('anno_metadata'))

terra_table_files <- cbind(
  anvil_file_tbl[, .(fnm=`pfb:file_name`, `pfb:file_ref`) |> transpose(make.names='fnm')],
  anno_meta_tbl [, .(fnm=basename(file),       file     ) |> transpose(make.names='fnm')]
)

terra_table_files

Registered S3 methods overwritten by 'AnVIL':
  method                         from    
  print.avworkflow_configuration AnVILGCP
  print.gcloud_sdk_result        AnVILGCP



GM20543_R2_001.fastq.gz,GM20502-16-2_R1_001.fastq.gz,HG02058_R2_001.fastq.gz,AFR.eQTL_FastQTL_results.nominal_pass.allAssociations.MAGE.v1.0.txt.gz,GM19028_R2_001.fastq.gz,GM12812_R1_001.fastq.gz,HG02513_R1_001.fastq.gz,GM12812_R2_001.fastq.gz,HG02643_R1_001.fastq.gz,GM18516_R1_001.fastq.gz,⋯,gerp_conservation_scores.homo_sapiens.GRCh38.bw,human_ancestor.fa.gz,loftee.sql,hg38.phyloP100way.bw,rename_chr.txt,1kG_cadd_subset.tsv.bgz,subset_cadd_cols2keep.txt,1kG_cadd_subset.tsv.bgz.tbi,unrename_chr.txt,homo_sapiens_vep_115_GRCh38.tar.gz
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
drs://drs.anv0:v2_904e31df-70a0-374f-9f30-d7117d26c4b4,drs://drs.anv0:v2_db413193-03e0-3044-bc0f-8506aff1ed6f,drs://drs.anv0:v2_3320e14d-48bf-30f2-a62f-7dd17f363fd1,drs://drs.anv0:v2_05efde6e-5b6d-3086-a89c-e4ff0faa98f3,drs://drs.anv0:v2_bd3276d4-e9d0-392e-9c71-5faf69c1c39d,drs://drs.anv0:v2_c1c91102-f136-3569-b4ab-65d17808de26,drs://drs.anv0:v2_5e76e62e-28df-3027-a5f5-fed3cdb4d6d8,drs://drs.anv0:v2_35100d70-059f-3be2-8d97-4ab19ca6a8f3,drs://drs.anv0:v2_8a575e36-5e88-382a-8ee5-1d33020b4d55,drs://drs.anv0:v2_93b7887f-cab3-356f-9189-b9693cdf96f2,⋯,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/gerp_conservation_scores.homo_sapiens.GRCh38.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/human_ancestor.fa.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/loftee.sql,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/hg38.phyloP100way.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/rename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/subset_cadd_cols2keep.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz.tbi,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/unrename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/homo_sapiens_vep_115_GRCh38.tar.gz


## Construct Terra data table specifying workflow inputs

Here we pull file references from the `anvil_file` and `anno_metadata` tables and organize them into a new, Terra data table. Each column of this new table represents an input to the Watershed-SV annotation workflow.

First, we collect the inputs that will be common to all runs.

In [4]:
shared_fields <- terra_table_files[
  ][, .(# Renaming and keeping only the needed files.
        #cadd_cache           = whole_genome_SNVs_inclAnno.tsv.gz,
        #cadd_cache_idx       = whole_genome_SNVs_inclAnno.tsv.gz.tbi,
        #cadd_cols2keep       = cadd_cols2keep.txt,
        cadd_cache           = `1kG_cadd_subset.tsv.bgz`,
        cadd_cache_idx       = `1kG_cadd_subset.tsv.bgz.tbi`,
        cadd_cols2keep       = subset_cadd_cols2keep.txt,
        cadd_anno_header     = cadd.hdr,
        chr_rename_file      = rename_chr.txt,
        chr_unrename_file    = unrename_chr.txt,
      
        gerp_bw              = gerp_conservation_scores.homo_sapiens.GRCh38.bw,
        human_ancestor_seq   = human_ancestor.fa.gz,
        phylocsf_db          = loftee.sql,
        phylop100_bw         = hg38.phyloP100way.bw,
        vep_cache_tar        = homo_sapiens_vep_115_GRCh38.tar.gz,

        filter_regions       = paste0(avstorage(),'/data/derived/regions_near_expression_genes.tsv'),
        filter_samples       = paste0(avstorage(),'/data/derived/samples_with_expression_data.txt')
      )
]

shared_fields

cadd_cache,cadd_cache_idx,cadd_cols2keep,cadd_anno_header,chr_rename_file,chr_unrename_file,gerp_bw,human_ancestor_seq,phylocsf_db,phylop100_bw,vep_cache_tar,filter_regions,filter_samples
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz.tbi,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/subset_cadd_cols2keep.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/cadd.hdr,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/rename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/unrename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/gerp_conservation_scores.homo_sapiens.GRCh38.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/human_ancestor.fa.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/loftee.sql,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/hg38.phyloP100way.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/homo_sapiens_vep_115_GRCh38.tar.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/derived/regions_near_expression_genes.tsv,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/derived/samples_with_expression_data.txt


Next, we use the shared input template to make a few rows. Each row has a different set of chromosomes, so you may choose to run the annotation workflow on all chromosomes or only a subset.

In [5]:
watershed_snv_run_tbl <- rbind(
  shared_fields[, c(
    watershed_snv_run='MAGE_1000G',
    vcfs = toJSON( terra_table_files[, unlist(.SD), .SDcols=patterns('1kGP.*.vcf.gz$')] ),
    .SD
  )],
  # At this section, choose the chromosomes of interest.
  # This example is run for chr 21 and chr 22.
  shared_fields[, c(
    watershed_snv_run='MAGE_1000G_chr22_chr21',
    vcfs = toJSON( terra_table_files[, unlist(.SD), .SDcols=patterns('1kGP_high_coverage_Illumina.chr2[12].*.vcf.gz$')] ),
    .SD
  )],
  # This example is run for chr 18, chr 19, and chr 20.
  shared_fields[, c(
    watershed_snv_run='MAGE_1000G_chr18_chr19_chr20',
    vcfs = toJSON( terra_table_files[, unlist(.SD), .SDcols=patterns('1kGP_high_coverage_Illumina.chr(18|19|20).*.vcf.gz$')] ),
    .SD
  )]
)

watershed_snv_run_tbl
# Note: The name of the first column determines the name of the terra table. So it will be named "watershed_snv_run."

watershed_snv_run,vcfs,cadd_cache,cadd_cache_idx,cadd_cols2keep,cadd_anno_header,chr_rename_file,chr_unrename_file,gerp_bw,human_ancestor_seq,phylocsf_db,phylop100_bw,vep_cache_tar,filter_regions,filter_samples
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
MAGE_1000G,"[""drs://drs.anv0:v2_b8f4e420-06f2-3024-b8b8-b6469e0fb5ef"",""drs://drs.anv0:v2_2379ffe3-3510-353f-acbf-7a5a71f73427"",""drs://drs.anv0:v2_e804b901-1ddf-363d-a72e-521e524b366d"",""drs://drs.anv0:v2_ccc7cdd3-0745-3c8e-b7f1-0be05c10a015"",""drs://drs.anv0:v2_f0b065c7-8e76-3805-9f56-51588fdcc17c"",""drs://drs.anv0:v2_c409c60c-2d84-352a-86d1-615c3a9ef0df"",""drs://drs.anv0:v2_477fd9e6-aade-3359-846f-5e44c6dd417e"",""drs://drs.anv0:v2_b4fcc3c1-551d-3b45-965a-3efbb551a56b"",""drs://drs.anv0:v2_92905946-966c-3a37-b725-35c77151d166"",""drs://drs.anv0:v2_6aaa7b02-eb62-314e-9e9c-bd97e28d0590"",""drs://drs.anv0:v2_cae79d39-6b65-3c56-9b87-2138ad313cad"",""drs://drs.anv0:v2_21c84537-2520-39a7-bef5-89f9752eb06b"",""drs://drs.anv0:v2_1c27ae9b-9f7b-336d-b605-955daa765f23"",""drs://drs.anv0:v2_be5f294c-233c-35de-92d0-61ad3ef622d7"",""drs://drs.anv0:v2_a0ca182a-ad8a-34cd-a151-63ffc16f1197"",""drs://drs.anv0:v2_0dbb2a75-04f5-341e-a30b-aeb64b1908c3"",""drs://drs.anv0:v2_ce8d7437-2e3f-3fac-8b8d-5bdcbbbccc5b"",""drs://drs.anv0:v2_a803b70e-f8e8-34cc-bf5a-8f59928b04f3"",""drs://drs.anv0:v2_cc590cb1-5be2-3b1b-9a0a-1d10126b137c"",""drs://drs.anv0:v2_59b09f4c-2744-3a2c-92d5-7554c3bef1ed"",""drs://drs.anv0:v2_1a81a00a-23ac-322c-a39c-bdf7ecfdafd9"",""drs://drs.anv0:v2_c6634514-7f58-3195-a128-f455e3943cc9"",""drs://drs.anv0:v2_e56fd786-640d-39ac-a8f8-61332d17f612""]",gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz.tbi,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/subset_cadd_cols2keep.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/cadd.hdr,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/rename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/unrename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/gerp_conservation_scores.homo_sapiens.GRCh38.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/human_ancestor.fa.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/loftee.sql,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/hg38.phyloP100way.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/homo_sapiens_vep_115_GRCh38.tar.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/derived/regions_near_expression_genes.tsv,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/derived/samples_with_expression_data.txt
MAGE_1000G_chr22_chr21,"[""drs://drs.anv0:v2_0dbb2a75-04f5-341e-a30b-aeb64b1908c3"",""drs://drs.anv0:v2_e56fd786-640d-39ac-a8f8-61332d17f612""]",gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/1kG_cadd_subset.tsv.bgz.tbi,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/subset_cadd_cols2keep.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/cadd.hdr,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/rename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/unrename_chr.txt,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/gerp_conservation_scores.homo_sapiens.GRCh38.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/human_ancestor.fa.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/loftee.sql,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/hg38.phyloP100way.bw,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/annotation/homo_sapiens_vep_115_GRCh38.tar.gz,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/derived/regions_near_expression_genes.tsv,gs://fc-c41bfd09-6c0b-47c3-a589-b91b317056c5/data/deriv

## Upload
`AnVILGCP::avtable_import()` is used to upload an R dataframe as a Terra data table.

In [6]:
avtable_import(watershed_snv_run_tbl, 'watershed_snv_run')

pageSize = 3 rows (1 pages)



page,from_row,to_row,job_id,status,message
<int>,<int>,<int>,<chr>,<chr>,<chr>
1,1,3,f8db59fa-3d28-4dbc-a1a6-e635fe0d6d50,Uploaded,NA


After running this notebook, check the data tab. There should be a new table called watershed_snv_run.

Scroll to the right to the vcfs column. Click the drs uri for one of the rows and verify that it is the intended file type, vcf.gz.